# Data Mining, Chapter 07 — Classification and trustworthy evaluation

This notebook is an **educational example with synthetic data**. Run cells from top to bottom. Never interpret its scores as operational performance.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from chapter07_examples import (toy_rows, split_gain, laplace_nb_demo, make_synthetic, split_by_customer, evaluate_pipeline, score_at_threshold, choose_threshold, FEATURES)
ROOT = Path.cwd()
print("Directory:", ROOT)

Directory: /mnt/data/_dm_ch07_author/downloads


## 1. Hand-checkable data and decision-tree impurity

In [2]:
toy = toy_rows()
print(toy.to_string(index=False))
print("Gini split:",split_gain(toy.new_device,toy.label))
print("Entropy split:",split_gain(toy.new_device,toy.label,"entropy"))

order_id  amount  new_device  failed_attempts  label
     T01     980           1                3      1
     T02     870           1                2      1
     T03     770           1                2      1
     T04     450           1                0      1
     T05     820           1                0      0
     T06     760           0                2      1
     T07     250           0                1      0
     T08     410           0                0      0
     T09     120           0                0      0
     T10     230           0                0      0
Gini split: {'parent': 0.5, 'children_weighted': 0.31999999999999984, 'gain': 0.18000000000000016}
Entropy split: {'parent': 1.0, 'children_weighted': 0.7219280948873623, 'gain': 0.2780719051126377}


## 2. Laplace-smoothed Naive Bayes

In [3]:
print("Naive Bayes worked example:", laplace_nb_demo())

Naive Bayes worked example: {'class1': 0.25510204081632654, 'class0': 0.02040816326530612, 'posterior': 0.9259259259259258}


## 3. Manual confusion matrix and alternative costs

In [4]:
y = [1]*20+[0]*180
scores = [.9]*12+[.1]*8+[.9]*18+[.1]*162
m = score_at_threshold(y,scores,threshold=.5)
print(m)
print("Cost for this classifier:", 8*m["fn"]+m["fp"])

{'tp': 12, 'fp': 18, 'fn': 8, 'tn': 162, 'accuracy': 0.87, 'balanced_accuracy': 0.75, 'precision': 0.4, 'recall': 0.6, 'f1': 0.48}
Cost for this classifier: 82


## 4. Generate and inspect independent-group synthetic records

In [5]:
data_file = ROOT / "fraud_events_synthetic.csv"
if not data_file.exists():make_synthetic(data_file)
df = pd.read_csv(data_file)
print("events:",len(df),"customers:",df.customer_id.nunique(),"positives:",df.fraud_flag.sum())
print(df.head(5).to_string(index=False))

events: 720 customers: 120 positives: 105
event_id customer_id  amount_toman_thousands  failed_attempts  new_device  night  prior_alerts  fraud_flag
 E000-00        C000                  552.52                0           0      0             0           0
 E000-01        C000                  606.72                2           1      0             0           1
 E000-02        C000                   97.75                0           1      1             0           0
 E000-03        C000                  881.32                0           0      1             0           1
 E000-04        C000                  305.62                0           0      0             0           0


## 5. Group-safe train, validation, test (do not shuffle independent rows)

In [6]:
train,validation,test = split_by_customer(df)
assert not (set(train.customer_id) & set(test.customer_id))
assert not (set(validation.customer_id) & set(test.customer_id))
assert not (set(train.customer_id) & set(validation.customer_id))
print({name:(len(part),round(float(part.fraud_flag.mean()),3),part.customer_id.nunique()) for name,part in [("train",train),("validation",validation),("test",test)]})

{'train': (402, 0.159, 67), 'validation': (138, 0.152, 23), 'test': (180, 0.111, 30)}


## 6. Model selection uses validation; final test is untouched

In [7]:
report = evaluate_pipeline(data_file)
print("Validation model comparison:")
for k,v in report["model_validation"].items():print(k,{m:round(z,4) for m,z in v.items()})
print("Selected on validation:",report["selected_model"],"threshold:",report["selected_threshold"])

Validation model comparison:
majority {'ap': 0.1522, 'roc_auc': 0.5, 'brier': 0.1291}
tree {'ap': 0.6259, 'roc_auc': 0.9023, 'brier': 0.0824}
logistic {'ap': 0.7104, 'roc_auc': 0.921, 'brier': 0.0782}
Selected on validation: logistic threshold: 0.05


## 7. What the one-time held-out test actually says

In [8]:
print("Held-out test:", {k:round(v,5) if isinstance(v,float) else v for k,v in report["test"].items()})
print("Baseline AP = test prevalence:",round(report["baseline_test_ap"],4))

Held-out test: {'roc_auc': 0.94937, 'average_precision': 0.868, 'brier': 0.03535, 'tp': 19, 'fp': 52, 'fn': 1, 'tn': 108, 'accuracy': 0.70556, 'balanced_accuracy': 0.8125, 'precision': 0.26761, 'recall': 0.95, 'f1': 0.41758}
Baseline AP = test prevalence: 0.1111


## 8. Cross-validation among development customers only

In [9]:
print("Group CV AP for logistic:", [round(x,4) for x in report["group_cv_logistic_AP"]])
print("Mean:",round(float(np.mean(report["group_cv_logistic_AP"])),4))

Group CV AP for logistic: [0.7495, 0.8228, 0.7335, 0.7364]
Mean: 0.7605


## 9. Cost-sensitive threshold selection works ONLY on validation labels

In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
m = make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=207)).fit(train[FEATURES],train.fraud_flag)
val_probability=m.predict_proba(validation[FEATURES])[:,1]
choice=choose_threshold(validation.fraud_flag,val_probability,fn_cost=8,fp_cost=1)
print({k:v for k,v in choice.items() if k!="costs"})

{'threshold': 0.05, 'validation_cost': 31}


## 10. Compare decision rules and keep the test sealed for selection

In [11]:
p=m.predict_proba(test[FEATURES])[:,1]
for t in [0.5,choice["threshold"]]:
    score=score_at_threshold(test.fraud_flag,p,t)
    print("threshold",t,"cost",8*score["fn"]+score["fp"],"TP,FP,FN,TN:",[score[k] for k in ("tp","fp","fn","tn")])

threshold 0.5 cost 51 TP,FP,FN,TN: [14, 3, 6, 157]


threshold 0.05 cost 60 TP,FP,FN,TN: [19, 52, 1, 108]


## 11. Learner activity: change the cost ratio on validation, not test

In [12]:
for fn_cost in [2,5,8,12]:
    pick=choose_threshold(validation.fraud_flag,val_probability,fn_cost=fn_cost,fp_cost=1)
    print(f"cost FN={fn_cost}: validation threshold={pick['threshold']:.3f}; validation cost={pick['validation_cost']}")

cost FN=2: validation threshold=0.425; validation cost=22


cost FN=5: validation threshold=0.050; validation cost=31


cost FN=8: validation threshold=0.050; validation cost=31


cost FN=12: validation threshold=0.050; validation cost=31


## 12. Final reproducibility notes

All data are synthetic. Do not remove `customer_id` from the split procedure. Changes to feature engineering should be fit **only on the training folds**. The reported test scores belong only to this generated dataset, seed and software environment.